In [0]:
import requests
import pandas as pd
import time
from datetime import timedelta
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
dbutils.widgets.text("bronze_catalog", "dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")


TABLE_NAME = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_historical_voivodeships_weather"


TARGET_START_DATE = pd.to_datetime("2020-01-01")
TARGET_END_DATE = pd.to_datetime("2024-12-31")
URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
NASA_HOURLY_PARAMS = "T2M,PRECTOTCORR,WS10M,TS,RH2M,T2MDEW"

In [0]:
voivodeships = {
    "Dolnośląskie": (51.10, 16.30),
    "Kujawsko-Pomorskie": (53.00, 18.60),
    "Lubelskie": (51.20, 23.00),
    "Lubuskie": (52.20, 15.30),
    "Łódzkie": (51.60, 19.40),
    "Małopolskie": (49.80, 20.00),
    "Mazowieckie": (52.20, 21.00),
    "Opolskie": (50.60, 17.90),
    "Podkarpackie": (49.90, 22.00),
    "Podlaskie": (53.10, 23.10),
    "Pomorskie": (54.20, 18.00),
    "Śląskie": (50.30, 19.00),
    "Świętokrzyskie": (50.80, 20.60),
    "Warmińsko-Mazurskie": (53.80, 20.80),
    "Wielkopolskie": (52.30, 17.00),
    "Zachodniopomorskie": (53.60, 15.60)
}

Implementation of idempotency

In [0]:
# pointer
state_pointers = {}
# idempotency
if spark.catalog.tableExists(TABLE_NAME):
    # Get the maximum date for each voivodeship
    df_state = spark.sql(f"""
        SELECT voivodeship, MAX(time) as max_time 
        FROM {TABLE_NAME} 
        GROUP BY voivodeship
    """)
    for row in df_state.collect():
        state_pointers[row['voivodeship']] = pd.to_datetime(row['max_time'])



In [0]:
total_regions = len(voivodeships)

weather_schema = StructType([
    StructField("voivodeship", StringType(), False), 
    StructField("lat", StringType(), False),
    StructField("lon", StringType(), False),
    StructField("time", StringType(), False),   
    StructField("temperature_2m", StringType(), False),
    StructField("precipitation", StringType(), False),
    StructField("wind_speed_10m", StringType(), False),
    StructField("soil_temp", StringType(), False),
    StructField("humidity_2m", StringType(), False),
    StructField("dew_point_2m", StringType(), False)
])


for index, (voivodeship, (lat, lon)) in enumerate(voivodeships.items(), 1):
    # Determine the start date for the specific voivodeship
    last_saved_time = state_pointers.get(voivodeship)
    
    if last_saved_time:
        # If data exists, start from the day after the last record
        current_start_date = last_saved_time + timedelta(days=1)
    else:
        # If no data exists, start from the default start date
        current_start_date = TARGET_START_DATE
          
    
    # Split the required period into chunks (by month) for safety
    # freq='M' generates month ends. We use this to avoid overloading the API with a full year at once.
    date_chunks = pd.date_range(start=current_start_date, end=TARGET_END_DATE, freq='ME').tolist()
    if TARGET_END_DATE not in date_chunks:
        date_chunks.append(TARGET_END_DATE)

    chunk_start = current_start_date

    for chunk_end in date_chunks:
        start_str = chunk_start.strftime('%Y%m%d')
        end_str = chunk_end.strftime('%Y%m%d')       
        api_params = {
            "parameters": NASA_HOURLY_PARAMS,
            "community": "AG",
            "longitude": lon,
            "latitude": lat,
            "start": start_str,
            "end": end_str,
            "format": "JSON"
        }

        chunk_data = []

        max_retries = 3
        for attempt in range(max_retries):
            response = requests.get(URL, params=api_params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                parameters = data.get("properties", {}).get("parameter", {})
                if not parameters:
                    break
                datetime_keys = list(parameters.get("T2M", {}).keys())
                for dt_key in datetime_keys:
                    hour_int = int(dt_key[-2:])
                    def get_val(param_name):
                        val = parameters.get(param_name, {}).get(dt_key)
                        return None if val == -999.0 else val
                    chunk_data.append({
                        "voivodeship": voivodeship,
                        "lat": lat,
                        "lon": lon,
                        "time": f"{dt_key[:4]}-{dt_key[4:6]}-{dt_key[6:8]} {dt_key[-2:]}:00:00",
                        "temperature_2m": get_val("T2M"),
                        "precipitation": get_val("PRECTOTCORR"),
                        "wind_speed_10m": get_val("WS10M"),
                        "soil_temp": get_val("TS"),
                        "humidity_2m": get_val("RH2M"),
                        "dew_point_2m": get_val("T2MDEW")
                    })
                break 
            elif response.status_code == 429:
                time.sleep(15)
            else:
                # print(f"API Error {response.status_code}. Skipping.")
                break              
        # SAVE CHUNK TO DATABASE

        if chunk_data:
            df_spark = spark.createDataFrame(chunk_data, schema=weather_schema)            
            (df_spark.write 
                .mode("append") 
                .partitionBy("voivodeship") 
                .format("delta") 
                .saveAsTable(TABLE_NAME)
            ) 
        # Shift the start of the next chunk by 1 day forward
        chunk_start = chunk_end + timedelta(days=1)
        time.sleep(0.1) # Protection against NASA API ban